# Historical experiment

Preserved for provenance only. This notebook has known methodological or portability issues; use the corrected notebook linked from the root README. Outputs have been cleared.

In [ ]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense ,Activation
from keras.optimizers import Adam

columns = (['duration','protocol_type','service','flag',
            'src_bytes','dst_bytes','land','wrong_fragment','urgent','hot'
,'num_failed_logins','logged_in','num_compromised','root_shell',
            'su_attempted','num_root','num_file_creations'
,'num_shells','num_access_files','num_outbound_cmds',
            'is_host_login','is_guest_login','count','srv_count','serror_rate'
,'srv_serror_rate','rerror_rate','srv_rerror_rate',
            'same_srv_rate','diff_srv_rate','srv_diff_host_rate',
            'dst_host_count','dst_host_srv_count'
,'dst_host_same_srv_rate','dst_host_diff_srv_rate',
            'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
            'dst_host_serror_rate'
,'dst_host_srv_serror_rate','dst_host_rerror_rate',
            'dst_host_srv_rerror_rate','outcome','level'])


df_train = pd.read_csv("/home/yesser/Downloads/intrusion/KDDTrain+.txt", header=None)
df_test = pd.read_csv("/home/yesser/Downloads/intrusion/KDDTest+.txt",  header=None)

df = pd.concat([df_train, df_test], ignore_index=True)
df.columns = columns
df.head()

In [ ]:
df.shape

In [ ]:
labels = []
for i in df["outcome"]:
    if i == "normal":
        labels.append(0)
    else: labels.append(1)
        
labels = np.array(labels)

In [ ]:
df = df.drop(["outcome"],axis=1)

In [ ]:
def make_dict(l):
    d = {}
    s = set(l)
    for e,i in enumerate(s):
        d[i] = e
    return d
        

In [ ]:
dict_protocol = make_dict(df['protocol_type'])
dict_service = make_dict(df['service'])
dict_flag = make_dict(df['flag'])

In [ ]:
dict_flag

In [ ]:
df['flag'] = df['flag'].map(dict_flag)
df['protocol_type'] = df['protocol_type'].map(dict_protocol)
df['service'] = df['service'].map(dict_service)

df_test['flag'] = df['flag'].map(dict_flag)
df_test['protocol_type'] = df['protocol_type'].map(dict_protocol)
df['service'] = df['service'].map(dict_service)

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

l = ['protocol_type', 'service', 'flag','level']

scaler = MinMaxScaler()

for col in df.columns:
    if col not in l:
        scaler.fit(np.array(df[col]).reshape(-1, 1))
        df[col] = scaler.transform(np.array(df[col]).reshape(-1, 1))
        df_test[col] = scaler.transform(np.array(df_test[col]).reshape(-1, 1))



In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df, labels, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Dense(128, input_shape=(df_train_data.shape[1],)))
model.add(Activation("relu"))
model.add(Dense(64))
model.add(Activation("relu"))
model.add(Dense(32))
model.add(Dense(1))
model.add(Activation("sigmoid"))

model.compile(optimizer=Adam(),
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

history = model.fit(X_train, y_train, epochs=5, batch_size=64, 
                    validation_data=(X_val, y_val))


In [ ]:
met = model.evaluate(X_test, y_test)
print("Accuracy : ",met[1])
print("Loss : ",met[0])

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = []
for p in model.predict(X_test):
    y_pred.append(round(p[0]))

cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
"""
#saving a model

model_json = model.to_json()
with open("model99"+".json", "w") as json_file:
    json_file.write(model_json)
model.save_weights("model99"+".h5")
print("Sauvegarde")"""

In [ ]:
"""
#loading a saved model

from tensorflow.keras.models import model_from_json

json_file = open("model99"+'.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
loaded_model.load_weights("model99"+".h5")

loaded_model.compile(optimizer=Adam(),
              loss="binary_crossentropy",
              metrics=["accuracy"])

met = loaded_model.evaluate(X_test, y_test)
print("Accuracy : ",met[1])
print("Loss : ",met[0])"""